In [3]:
#2.1

import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline as SklearnPipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, f1_score


# Load and preprocess dataset
df = pd.read_json("C:\\Users\\kgour\\OneDrive\\Documents\\School\\ITNP\\Research-Document-Mining\\datasets\\research_abstracts.json")
df = df.drop(columns=["DOI"])
df = df.dropna(subset=["Abstract", "Label"])
df = df.drop_duplicates()

X = df["Abstract"].astype(str)
y = df["Label"]


# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# Define four machine learning models
models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, random_state=42
    )
}


# Train and evaluate four models
results = []
trained_models = {}

for name, classifier in models.items():

    model = SklearnPipeline([
        ("tfidf", TfidfVectorizer(
            max_features=5000, ngram_range=(1, 2)
        )),
        ("classifier", classifier)
    ])

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    macro_f1 = f1_score(y_test, predictions, average="macro")
    weighted_f1 = f1_score(y_test, predictions, average="weighted")

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1
    })

    trained_models[name] = model

    print("\n", name)
    print(classification_report(
        y_test, predictions, zero_division=0
    ))


# Display model comparison
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="Macro F1", ascending=False
)

print("\nMODEL COMPARISON")
print(results_df.to_string(index=False))


# Select best model before SMOTE
best_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_name]

print("\nBest model before SMOTE:", best_name)


#2.2

# Apply SMOTE to the best model
smote_model = ImbPipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000, ngram_range=(1, 2)
    )),
    ("smote", SMOTE(random_state=42)),
    ("classifier", models[best_name])
])

smote_model.fit(X_train, y_train)

smote_predictions = smote_model.predict(X_test)

smote_accuracy = accuracy_score(y_test, smote_predictions)
smote_macro_f1 = f1_score(
    y_test, smote_predictions, average="macro"
)
smote_weighted_f1 = f1_score(
    y_test, smote_predictions, average="weighted"
)


# Compare performance before and after SMOTE
comparison = pd.DataFrame([
    {
        "Configuration": "Without SMOTE",
        "Accuracy": accuracy_score(
            y_test, best_model.predict(X_test)
        ),
        "Macro F1": f1_score(
            y_test, best_model.predict(X_test), average="macro"
        ),
        "Weighted F1": f1_score(
            y_test, best_model.predict(X_test), average="weighted"
        )
    },
    {
        "Configuration": "With SMOTE",
        "Accuracy": smote_accuracy,
        "Macro F1": smote_macro_f1,
        "Weighted F1": smote_weighted_f1
    }
])

print("\nSMOTE PERFORMANCE COMPARISON")
print(comparison.to_string(index=False))

print("\nCLASSIFICATION REPORT AFTER SMOTE")
print(classification_report(
    y_test, smote_predictions, zero_division=0
))


# Select the best overall configuration
if smote_macro_f1 > results_df.iloc[0]["Macro F1"]:
    final_model = smote_model
    final_name = best_name + " with SMOTE"
else:
    final_model = best_model
    final_name = best_name + " without SMOTE"


# Save the best model
joblib.dump(final_model, "best_model.joblib")

print("\nBEST OVERALL CONFIGURATION:", final_name)
print("Best model saved as best_model.joblib")



 Multinomial Naive Bayes
              precision    recall  f1-score   support

          ai       1.00      0.13      0.23        47
   classical       0.81      1.00      0.90       178

    accuracy                           0.82       225
   macro avg       0.91      0.56      0.56       225
weighted avg       0.85      0.82      0.76       225


 Linear SVM
              precision    recall  f1-score   support

          ai       0.85      0.62      0.72        47
   classical       0.91      0.97      0.94       178

    accuracy                           0.90       225
   macro avg       0.88      0.79      0.83       225
weighted avg       0.89      0.90      0.89       225


 Decision Tree
              precision    recall  f1-score   support

          ai       0.68      0.64      0.66        47
   classical       0.91      0.92      0.91       178

    accuracy                           0.86       225
   macro avg       0.79      0.78      0.79       225
weighted avg       